# Importation et chargement des donnees

In [ ]:
# Importation des librairies
import pandas as pd
import numpy as np
from datetime import datetime

# model_selection : le module de sklearn qui permet de découper, tester et optimiser les modèles.
from sklearn.model_selection import train_test_split

#  les transformateurs sont des objets utilisés dans les pipelines pour transformer les données
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, FunctionTransformer, OneHotEncoder
from sklearn.impute import SimpleImputer    # Remplace les valeurs manquantes par une valeur calculée ou définie.
from sklearn.compose import ColumnTransformer  # permet d’appliquer différentes transformations à différentes colonnes
from sklearn.pipeline import Pipeline    # permet de chaîner plusieurs étapes de traitement et de modélisation
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


# ==========================================
# 1. LOGIQUE DE TRANSFORMATION
# ==========================================

# Fonction pour la conversion des montants
def transform_montant(X, taux=130):
    """Convertit montant en Gdes.
    - Si contient 'US' => valeur USD * 130
    - Sinon si contient k => valeur milier*1000
    Sinon => valeur deja en Gdes
    - Valeurs non convertibles => NaN
    """
    X = pd.DataFrame(X).copy()
    col = X.columns[0]
    s = X[col]

    # Transformer en string, strip + uppercase
    s_str = s.astype(str).str.strip()
    s_up = s_str.str.upper()

    # Detecter USD
    is_usd = s_up.str.contains("US", na=False)

    # Detecter k (miliers)
    is_k = s_up.str.contains("K", na=False)

    # Retirer "US", ',', 'K' sans regex, puis nettoyer les espaces
    s_clean = (
        s_up.str.replace("USD", "", regex=False)
        .str.replace("K", "", regex=False)
        .str.replace("HTG", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    # Convertir en float
    num = pd.to_numeric(s_clean, errors="coerce")

    # Conversion: USD -> Gdes * taux, sinon garder tel quel
    out = num.where(~is_usd, num * taux)

    # Conversion: k -> *1000
    out = out.where(~is_k, out * 1000)

    return pd.DataFrame(out, columns=[col])


# Fonction pour le nettoyage des dates
def transform_date_features(X):
    """
    Parser robuste et extraction de variables temporelles.
    Gere les 'Inconnu', formats invalides et cree jour/mois.
    """
    X = pd.DataFrame(X).copy()
    col = X.columns[0]

    # Nettoyage simple des placeholders
    s = (
        X[col].astype(str).str.strip()
        .replace({'N/A': None, 'Inconnu': None, 'nan': None})
    )

    # Parsing des dates avec gestion des erreurs
    dt = pd.to_datetime(s, errors='coerce', dayfirst=True)

    # Creation des variables
    features = pd.DataFrame(index=X.index)
    features['jour_semaine'] = dt.dt.weekday  # 0=Lundi, 6=Dimanche
    features['mois'] = dt.dt.month
    features['jour_mois'] = dt.dt.day
    features['date_invalide'] = dt.isna().astype(int)

    return features


# Fonction pour le nettoyage de la devise
def transform_device(X):
    """
    Nettoyer la colonne Devise en remplaçant USD par HTG
    """
    X = pd.DataFrame(X).copy()
    col = X.columns[0]
    
    s_clean = (
        X[col]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace("USD", "HTG")
    )
    
    return pd.DataFrame(s_clean, columns=[col])


# Fonction pour convertir l'anciennete en jours
def transform_anciennete_en_jours(X):
    """
    Convertir l'anciennete en jours
    """
    X = pd.DataFrame(X).copy()
    col = X.columns[0]
    s = X[col].astype(str).str.strip().str.lower()

    # Detection des unites
    is_years = (
        s.str.contains('year', na=False) |
        s.str.contains('years', na=False)
    )

    is_months = (
        s.str.contains('month', na=False) |
        s.str.contains('months', na=False) |
        s.str.contains('m', na=False)
    )

    # Nettoyage pour ne garder que le chiffre
    # On retire toutes les lettres et on gere la virgule
    s_clean = (
        s.str.replace(r'[a-z]', '', regex=True)
        .str.replace(',', '', regex=False)
        .str.strip()
    )
    num = pd.to_numeric(s_clean, errors='coerce')

    # Application des coefficients de conversion
    day = np.select(
        condlist=[is_years, is_months],
        choicelist=[num * 365, num * 30],
        default=num
    )

    return pd.DataFrame(day, columns=[col])


# Fonction pour la normalisation des villes
def normaliser_ville(X):
    X = pd.DataFrame(X).copy()
    col = X.columns[0]
    
    # Nettoyage de base
    s_clean = X[col].astype(str).str.strip().str.lower()

    # Dictionnaire de normalisation
    mapping = {
        'p-au-p': 'Port-au-Prince',
        'port au prince': 'Port-au-Prince',
        'pap': 'Port-au-Prince',
        'port-au-prince': 'Port-au-Prince',
        'jacmel': 'Jacmel',
        'cap': 'Cap-Haïtien',
        'cap haitien': 'Cap-Haïtien',
        'cap-haïtien': 'Cap-Haïtien',
        'hin': 'Hinche',
        'hinche': 'Hinche',
        'gonaives': 'Gonaïves',
        'gonaïves': 'Gonaïves',
        'cayes': 'Les Cayes',
        'les cayes': 'Les Cayes'
    }

    # Application du remplacement
    s_clean = s_clean.replace(mapping)

    # On remplace les chaines vides ou 'nan' par 'Inconnu'
    s_clean = s_clean.replace(['', 'nan', 'n/a', 'none'], 'Inconnu')

    return pd.DataFrame(s_clean, columns=[col])


# Fonction pour le nettoyage de l'age
def transform_age(X):
    """
    Nettoyer la colonne age en convertissant en numerique et en gerant les valeurs aberrantes    
    """
    X = pd.DataFrame(X).copy()
    col = X.columns[0]
    s = X[col]

    # Nettoyage de base
    s_clean = s.astype(str).str.strip()
    num = pd.to_numeric(s_clean, errors='coerce')

    # je vais clipper les ages entre 18 et 90 ans
    num = num.clip(lower=18, upper=90)

    return pd.DataFrame(num, columns=[col])


# Fonction pour le nettoyage du revenu
def transform_revenu_or_dette(X):
    """
    Nettoyer la colonne RevenuMensuel_raw ou Dette_raw 
    en convertissant en numerique et en gerant les valeurs manquantes    
    """
    X = pd.DataFrame(X).copy()
    col = X.columns[0]
    s = X[col]

    # Nettoyage de base et remplace les virgules
    s_clean = (
        s.astype(str).str.strip()
        .str.replace(',', '', regex=False)
    )

    # Convertir en float
    num = pd.to_numeric(s_clean, errors='coerce')

    return pd.DataFrame(num, columns=[col])


# Fonction pour la normalisation de Employe
def normaliser_employe(X):
    """
    Normaliser la colonne Employe en gerant les valeurs manquantes
    """
    X = pd.DataFrame(X).copy()
    col = X.columns[0]
    s = X[col]

    # Nettoyage de base
    s_clean = s.astype(str).str.strip().str.lower()

    # Dictionnaire de normalisation
    mapping = {
        'yes': 'Oui',
        'no': 'Non',
    }

    # Application du remplacement
    s_clean = s_clean.replace(mapping)

    # On remplace les chaines vides ou 'nan' par 'Inconnu'
    s_clean = s_clean.replace(['', 'nan', 'n/a', 'none'], 'Inconnu')

    # Mettre en majuscule la premiere lettre
    s_clean = s_clean.str.title()

    # Remplace les oui par 1 et les non par 0
    s_clean = s_clean.map({'Oui': 1, 'Non': 0, 'Inconnu': np.nan})

    # Convertir en float
    s_clean_float = pd.to_numeric(s_clean, errors='coerce')

    return pd.DataFrame(s_clean_float, columns=[col])


# Fonction pour traiter les Device
def normaliser_other(X):
    """
    Normaliser la colonne Device en gerant les valeurs manquantes
    """
    X = pd.DataFrame(X).copy()
    col = X.columns[0]
    s = X[col]

    # Nettoyage de base
    s_clean = s.astype(str).str.strip().str.lower()

    # On remplace les chaines vides ou 'nan' par 'Inconnu'
    s_clean = s_clean.replace(['', 'nan', 'n/a', 'none'], 'Inconnu')

    # Mettre en majuscule la premiere lettre
    s_clean = s_clean.str.title()

    return pd.DataFrame(s_clean, columns=[col])

# Fonction pour renommer les colonnes apres pretraitement
def rename_columns(X):
    """
        Renommer les colonnes de X pour eviter les confusions entre les brutes et les pretraitees
    """
    X = pd.DataFrame(X).copy()
    _pour_renommer = {
        'Montant_raw': 'Montant_HTG',
        'AncienneteCompte_raw': 'Anciennete_Jours',
        'RevenuMensuel_raw': 'Revenu_Mensuel',
        'Dette_raw': 'Dette',
        'Age': 'Age_Clean',
        'Employe': 'Employe_Statut',
        'Ville_raw': 'Ville_Clean',
        'Device': 'Device_Clean'
    }
    X = X.rename(columns=_pour_renommer)
    return X

# ==========================================
# CHARGEMENT ET PREPARATION DES DONNEES
# ==========================================

# Chargements des donnees
df = pd.read_excel('dataset_pretraitement_fraude.xlsx')

# Suppression les doublons
df = df.drop_duplicates()

col_exclude = ['TransactionID', 'ClientID', 'Commentaire']

col_todo_exclude = [col for col in df.columns if col.startswith('TODO_')]

X = df.drop(columns=col_exclude + col_todo_exclude)

y = df['Fraude']

# renommer les colonnes pour eviter les confusions entre les brutes et les pretraitees
X = rename_columns(X)


# ==========================================
# EXPORTATION DES DONNEES PRETRAITEES
# ==========================================
def export_data_clean():
    """
        Exporte un fichier Excel avec les donnees pretraitees
    """
    
    # Creer une copie de X pour le nettoyage
    X_clean_excel = X.copy()

    # Application des transformations
    X_clean_excel['Montant_HTG'] = transform_montant(X[['Montant_raw']])
    X_clean_excel['Anciennete_Jours'] = transform_anciennete_en_jours(X[['AncienneteCompte_raw']])
    X_clean_excel['Revenu_Mensuel'] = transform_revenu_or_dette(X[['RevenuMensuel_raw']])
    X_clean_excel['Dette'] = transform_revenu_or_dette(X[['Dette_raw']])
    X_clean_excel['Age_Clean'] = transform_age(X[['Age']])
    X_clean_excel['Employe_Statut'] = normaliser_employe(X[['Employe']])
    X_clean_excel['Ville_Clean'] = normaliser_ville(X[['Ville_raw']])
    X_clean_excel['Device_Clean'] = normaliser_other(X[['Device']])

    # Ajoutons la colonne cible a la fin
    X_clean_excel['Fraude_Cible'] = y
    X_clean_excel = X_clean_excel.drop(
        columns=[
            'Employe'
        ]
    )


    # Supprime les autres colonnes brutes
    colonnes_finales = [col for col in X_clean_excel.columns if not col.endswith('_raw')]
    df_final_export = X_clean_excel[colonnes_finales]

    # Exportation vers Excel
    df_final_export.to_excel("dataset_fraude_nettoye_final.xlsx", index=False)

    print("Ficheir excel cree avec succes !")


# ==========================================
# PIPELINE DE PRETRAITEMENT + MODELE
# ==========================================
niveau_etude_order = [
    'Primaire',
    'Secondaire',
    'Doctorat',
    'Master',
    'Licence',
    'Inconnu',
]

# On definit les colonnes pour eviter de se perdre
preprocessing_pipeline = ColumnTransformer(
    transformers=[
        # 1. Montant : Transform, Impute, Scale
        ('num_montant', Pipeline([
            ('convert', FunctionTransformer(transform_montant)),
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), ['Montant_raw']),

        # Dans ton preprocessing_pipeline :
        ('temporel', Pipeline([
            ('extract', FunctionTransformer(transform_date_features)),
            # Gerer les NaT (dates invalides/Inconnu) par la valeur la plus fréquente
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('scaler', StandardScaler())
        ]), ['DateTransaction_raw']),

        # 2. Anciennete : Transform, Impute, Scale
        ('num_anciennete', Pipeline([
            ('convert', FunctionTransformer(transform_anciennete_en_jours)),
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), ['AncienneteCompte_raw']),

        # 3. Revenu & Dette : Transform, Impute, Scale j'utilise la meme fonction pour les deux
        ('num_revenu', Pipeline([
            ('convert', FunctionTransformer(transform_revenu_or_dette)),
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), ['RevenuMensuel_raw']),

        ('num_dette', Pipeline([
            ('convert', FunctionTransformer(transform_revenu_or_dette)),
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), ['Dette_raw']),

        # 4. Age : Transform, Impute, Scale
        ('num_age', Pipeline([
            ('convert', FunctionTransformer(transform_age)),
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), ['Age']),

        # 5. Employe j'ai deja converti en 0/1/NaN : Impute
        ('num_employe', Pipeline([
            ('convert', FunctionTransformer(normaliser_employe)),
        ]), ['Employe']),

        # 6. Categories : Normaliser, OneHotEncoder
        ('cat_ville', Pipeline([
            ('norm', FunctionTransformer(normaliser_ville)),
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), ['Ville_raw']),

        ('cat_autres', Pipeline([
            ('norm', FunctionTransformer(normaliser_other)),
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), ['Device']),

        # TypeMarchand est une variable cat -> OneHotEncode apres normalisation
        ('cat_type_marchand', Pipeline([
            ('norm', FunctionTransformer(normaliser_other)),
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), ['TypeMarchand']),

        # Canal est cat -> OneHotEncode apres normalisation
        ('cat_canal', Pipeline([
            ('norm', FunctionTransformer(normaliser_other)),
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), ['Canal']),

        ('cat_niveau_etude', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ord', OrdinalEncoder(categories=[niveau_etude_order], handle_unknown='use_encoded_value', unknown_value=-1))
        ]), ['NiveauEtude']),

        # 7. NbTrans_24h : Impute, Scale
        ('num_nbtrans_24h', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), ['NbTrans_24h']),

        # 8. StatutMarital : Impute, OneHotEncoder
        ('cat_statutmarital', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), ['StatutMarital']),

        # 9. Devise_indiquee : Normaliser, Impute, OneHotEncoder
        ('cat_devise', Pipeline([
            ('norm', FunctionTransformer(transform_device)),
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), ['Devise_indiquee']),
    ]
)

full_model = Pipeline([
    ('pretraitement', preprocessing_pipeline),
    ('modele', LogisticRegression(max_iter=1000))
])

# ==========================================
# 4. ENTRAINEMENT ET EVALUATION
# ==========================================

# Decoupage en train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Entrainement du modele
full_model.fit(X_train, y_train)

# Prediction sur le jeu de test
y_pred = full_model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


<>:361: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
<>:361: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?


TypeError: 'tuple' object is not callable

In [19]:
export_data_clean()

Ficheir excel cree avec succes !
